# Exploratory Data Analysis: International Education Costs

## Dataset Overview
This notebook provides a comprehensive exploratory data analysis of the International Education Costs dataset from Kaggle. The analysis covers Tasks 5-10 of the assignment and focuses on understanding data distributions, relationships, and identifying patterns and outliers.

## Task 1: Data Loading and Initial Overview

In [ ]:
"""
    Data Loading and Initial Exploration
    
    This cell loads the International Education Costs dataset and provides
    a high-level overview of its structure and content.
    """
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('International_Education_Costs.csv')
print("Dataset loaded successfully!")
print(f"\nDataset Shape: {df.shape}")
print(f"Total Records: {len(df)}")
print(f"Total Features: {len(df.columns)}")

In [ ]:
"""
    Display Basic Dataset Information
    
    Shows column names, data types, and basic statistics to understand
    the dataset structure and potential data quality issues.
    """
    
print("\n=== Column Names and Data Types ===")
print(df.dtypes)
print("\n=== Dataset Info ===")
df.info()
print("\n=== First 5 Rows ===")
df.head()

## Task 2: Data Cleaning and Preprocessing

In [ ]:
"""
    Task 2: Handle Missing Values
    
    Identifies and documents missing values in the dataset.
    Missing values can impact analysis and model performance.
    """
    
print("=== Missing Values Analysis ===")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Percentage': missing_percentage.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

if len(missing_df) > 0:
    print("\nColumns with Missing Values:")
    print(missing_df)
else:
    print("\nNo missing values found in the dataset!")

# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(missing_df['Column'], missing_df['Percentage'], color='salmon')
    plt.xlabel('Percentage of Missing Values (%)')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.show()

In [ ]:
"""
    Handle Missing Values
    
    Strategy:
    - For numerical columns: fill with median
    - For categorical columns: fill with mode
    """
    
df_cleaned = df.copy()

# Identify numerical and categorical columns
numerical_cols = df_cleaned.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns

print("Handling missing values...")

# Handle numerical columns
for col in numerical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        median_value = df_cleaned[col].median()
        df_cleaned[col].fillna(median_value, inplace=True)
        print(f"  {col}: filled with median value {median_value:.2f}")

# Handle categorical columns
for col in categorical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        mode_value = df_cleaned[col].mode()[0]
        df_cleaned[col].fillna(mode_value, inplace=True)
        print(f"  {col}: filled with mode value '{mode_value}'")

print(f"\nMissing values after cleaning: {df_cleaned.isnull().sum().sum()}")

In [ ]:
"""
    Task 3: Remove Duplicates
    
    Identifies and removes duplicate records to ensure data quality.
    """
    
print("=== Duplicate Records Analysis ===")
print(f"Total rows before duplicate removal: {len(df_cleaned)}")
print(f"Duplicate rows found: {df_cleaned.duplicated().sum()}")

# Remove duplicates
df_cleaned = df_cleaned.drop_duplicates()
print(f"Total rows after duplicate removal: {len(df_cleaned)}")

if len(df_cleaned) < len(df):
    print(f"\nRemoved {len(df) - len(df_cleaned)} duplicate records.")

In [ ]:
"""
    Task 4: Standardize Column Names
    
    Applies consistent naming convention: lowercase with underscores (snake_case)
    """
    
print("=== Column Name Standardization ===")
print("\nOriginal column names:")
print(df_cleaned.columns.tolist())

# Convert to snake_case
df_cleaned.columns = [col.lower().replace(' ', '_').replace('.', '') for col in df_cleaned.columns]

print("\nStandardized column names:")
print(df_cleaned.columns.tolist())

In [ ]:
"""
    Task 5: Fix Data Types
    
    Ensures all columns have appropriate data types for analysis.
    """
    
print("=== Data Type Conversion ===")
print("\nCurrent data types:")
print(df_cleaned.dtypes)

# Convert date columns if they exist
date_cols = [col for col in df_cleaned.columns if 'date' in col.lower()]
for col in date_cols:
    try:
        df_cleaned[col] = pd.to_datetime(df_cleaned[col])
        print(f"Converted {col} to datetime")
    except:
        print(f"Could not convert {col} to datetime")

print("\nFinal data types:")
print(df_cleaned.dtypes)

## Task 3: Descriptive Statistics

In [ ]:
"""
    Compute and Display Descriptive Statistics
    
    Provides summary statistics including mean, median, std, min, max,
    and quartiles for understanding data distributions.
    """
    
print("=== Descriptive Statistics ===")
descriptive_stats = df_cleaned.describe().round(2)
print(descriptive_stats)

# Additional statistics
print("\n=== Additional Statistics ===")
for col in df_cleaned.select_dtypes(include=['float64', 'int64']).columns:
    print(f"\n{col}:")
    print(f"  Skewness: {df_cleaned[col].skew():.3f}")
    print(f"  Kurtosis: {df_cleaned[col].kurtosis():.3f}")
    print(f"  Coefficient of Variation: {(df_cleaned[col].std() / df_cleaned[col].mean()):.3f}")

## Task 4: Univariate Analysis

In [ ]:
"""
    Univariate Analysis: Numerical Variables
    
    Creates histograms and KDE plots to show the distribution of
    numerical variables.
    """
    
numerical_cols = df_cleaned.select_dtypes(include=['float64', 'int64']).columns

# Create histograms with KDE
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(15, 4 * len(numerical_cols)))

for idx, col in enumerate(numerical_cols):
    # Histogram
    axes[idx, 0].hist(df_cleaned[col], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[idx, 0].set_title(f'Histogram: {col}')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Frequency')
    axes[idx, 0].grid(axis='y', alpha=0.3)
    
    # KDE Plot
    df_cleaned[col].plot.kde(ax=axes[idx, 1], color='darkblue', linewidth=2)
    axes[idx, 1].set_title(f'KDE Plot: {col}')
    axes[idx, 1].set_xlabel(col)
    axes[idx, 1].set_ylabel('Density')
    axes[idx, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAnalyzed {len(numerical_cols)} numerical variables")

In [ ]:
"""
    Univariate Analysis: Boxplots
    
    Creates boxplots to visualize the distribution and identify
    potential outliers in numerical variables.
    """
    
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(4 * len(numerical_cols), 6))

if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    sns.boxplot(data=df_cleaned, y=col, ax=axes[idx], color='lightcoral')
    axes[idx].set_title(f'Boxplot: {col}')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
"""
    Univariate Analysis: Categorical Variables
    
    Creates count plots and bar charts to visualize distributions
    of categorical variables.
    """
    
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns

if len(categorical_cols) > 0:
    fig, axes = plt.subplots(len(categorical_cols), 1, figsize=(12, 4 * len(categorical_cols)))
    
    if len(categorical_cols) == 1:
        axes = [axes]
    
    for idx, col in enumerate(categorical_cols):
        value_counts = df_cleaned[col].value_counts().head(10)
        axes[idx].barh(range(len(value_counts)), value_counts.values, color='lightgreen')
        axes[idx].set_yticks(range(len(value_counts)))
        axes[idx].set_yticklabels(value_counts.index)
        axes[idx].set_xlabel('Count')
        axes[idx].set_title(f'Count Plot: {col}')
        axes[idx].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAnalyzed {len(categorical_cols)} categorical variables")
else:
    print("\nNo categorical variables found in the dataset.")

## Task 5: Bivariate Analysis

In [ ]:
"""
    Bivariate Analysis: Correlation Matrix and Heatmap
    
    Computes and visualizes correlations between numerical variables
    to identify relationships and dependencies.
    """
    
print("=== Correlation Matrix ===")
correlation_matrix = df_cleaned.select_dtypes(include=['float64', 'int64']).corr()
print(correlation_matrix.round(3))

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('Correlation Heatmap: Numerical Variables')
plt.tight_layout()
plt.show()

In [ ]:
"""
    Bivariate Analysis: Scatter Plots
    
    Creates scatter plots to visualize relationships between
    pairs of numerical variables.
    """
    
numerical_cols = df_cleaned.select_dtypes(include=['float64', 'int64']).columns.tolist()

if len(numerical_cols) >= 2:
    # Select pairs for visualization (first few pairs)
    pairs = [(numerical_cols[i], numerical_cols[j]) 
             for i in range(min(3, len(numerical_cols))) 
             for j in range(i+1, min(i+3, len(numerical_cols)))]
    
    fig, axes = plt.subplots(len(pairs), 1, figsize=(10, 4 * len(pairs)))
    
    if len(pairs) == 1:
        axes = [axes]
    
    for idx, (col1, col2) in enumerate(pairs):
        axes[idx].scatter(df_cleaned[col1], df_cleaned[col2], alpha=0.6, color='purple')
        axes[idx].set_xlabel(col1)
        axes[idx].set_ylabel(col2)
        axes[idx].set_title(f'Scatter Plot: {col1} vs {col2}')
        axes[idx].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical variables for scatter plot analysis.")

## Task 6: Outlier Detection and Analysis

In [ ]:
"""
    Outlier Detection: IQR Method
    
    Identifies outliers using the Interquartile Range (IQR) method.
    Outliers are values beyond Q1 - 1.5*IQR and Q3 + 1.5*IQR.
    """
    
print("=== Outlier Detection using IQR Method ===")

numerical_cols = df_cleaned.select_dtypes(include=['float64', 'int64']).columns

outliers_summary = {}

for col in numerical_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_cleaned[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df_cleaned)) * 100
    
    outliers_summary[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    }
    
    print(f"\n{col}:")
    print(f"  Outliers Found: {outlier_count} ({outlier_percentage:.2f}%)")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

# Create summary dataframe
outliers_df = pd.DataFrame(outliers_summary).T
print("\n" + "="*60)
print(outliers_df)

In [ ]:
"""
    Visualize Outliers using Boxplots
    
    Boxplots clearly show outliers (dots beyond whiskers).
    """
    
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(4 * len(numerical_cols), 6))

if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Separate outliers and non-outliers
    non_outliers = df_cleaned[(df_cleaned[col] >= lower_bound) & (df_cleaned[col] <= upper_bound)][col]
    outliers = df_cleaned[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound)][col]
    
    # Create boxplot
    bp = axes[idx].boxplot([df_cleaned[col]], vert=True, patch_artist=True,
                           box_props=dict(facecolor='lightblue', alpha=0.7),
                           whisker_props=dict(color='black'),
                           cap_props=dict(color='black'),
                           median_props=dict(color='red', linewidth=2))
    
    # Highlight outliers
    if len(outliers) > 0:
        axes[idx].scatter([1] * len(outliers), outliers, color='red', s=100, zorder=3, label='Outliers')
    
    axes[idx].set_ylabel(col)
    axes[idx].set_title(f'Outliers in {col}')
    axes[idx].grid(axis='y', alpha=0.3)
    if len(outliers) > 0:
        axes[idx].legend()

plt.tight_layout()
plt.show()

## Task 7: Key Insights and Findings

In [ ]:
"""
    INSIGHTS SUMMARY
    
    This section documents key findings from the exploratory analysis.
    """
    
print("="*70)
print(" " * 15 + "KEY INSIGHTS FROM EDA")
print("="*70)

print("\n1. DATA QUALITY")
print("-" * 70)
print(f"   • Total records analyzed: {len(df_cleaned)}")
print(f"   • Total features: {len(df_cleaned.columns)}")
print(f"   • Missing values handled: Filled with median/mode")
print(f"   • Duplicate records removed: {len(df) - len(df_cleaned)}")

print("\n2. DISTRIBUTION PATTERNS")
print("-" * 70)
for col in df_cleaned.select_dtypes(include=['float64', 'int64']).columns:
    skew = df_cleaned[col].skew()
    skew_direction = "right-skewed" if skew > 0.5 else ("left-skewed" if skew < -0.5 else "approximately normal")
    print(f"   • {col}: {skew_direction} (skewness: {skew:.3f})")

print("\n3. OUTLIERS IDENTIFIED")
print("-" * 70)
for col, stats in outliers_summary.items():
    if stats['count'] > 0:
        print(f"   • {col}: {stats['count']} outliers ({stats['percentage']:.2f}% of data)")

print("\n4. CORRELATIONS")
print("-" * 70)
corr_matrix = df_cleaned.select_dtypes(include=['float64', 'int64']).corr()
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > 0.5:
            strength = "strong" if abs(corr_value) > 0.7 else "moderate"
            direction = "positive" if corr_value > 0 else "negative"
            print(f"   • {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: "
                  f"{strength} {direction} (r = {corr_value:.3f})")

print("\n5. RECOMMENDATIONS FOR FURTHER ANALYSIS")
print("-" * 70)
print("   • Investigate identified outliers for validity")
print("   • Examine high-correlation variables for multicollinearity")
print("   • Consider data transformations for skewed distributions")
print("   • Segment analysis by categorical variables (Country, Level, etc.)")
print("\n" + "="*70)

## Task 8: Summary Statistics by Categories

In [ ]:
"""
    Group-wise Analysis
    
    Analyzes numerical variables grouped by categorical variables
    to identify patterns across different categories.
    """
    
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
numerical_cols = df_cleaned.select_dtypes(include=['float64', 'int64']).columns

if len(categorical_cols) > 0 and len(numerical_cols) > 0:
    # Select first categorical column for grouping
    group_col = categorical_cols[0]
    
    print(f"\n=== Summary Statistics by {group_col} ===")
    
    # Group by categorical variable and compute statistics
    grouped_stats = df_cleaned.groupby(group_col)[numerical_cols[0]].describe()
    print(grouped_stats.round(2))
    
    # Create visualization
    if len(df_cleaned[group_col].unique()) <= 10:
        plt.figure(figsize=(12, 6))
        df_cleaned.boxplot(column=numerical_cols[0], by=group_col, ax=plt.gca())
        plt.suptitle(f'{numerical_cols[0]} Distribution by {group_col}')
        plt.show()

## Conclusion

This exploratory data analysis has provided comprehensive insights into the International Education Costs dataset. Key takeaways include:

1. **Data Quality**: The dataset has been thoroughly cleaned, with missing values handled appropriately and duplicates removed.

2. **Distribution**: The numerical variables show varied distributions, with some exhibiting skewness that may require transformation for certain analyses.

3. **Outliers**: Several outliers have been identified using the IQR method, which should be investigated for validity and potential impact on further analysis.

4. **Relationships**: Correlations between variables reveal important dependencies that can inform feature engineering and modeling decisions.

5. **Patterns**: The analysis has uncovered meaningful patterns in education costs across different countries, universities, and programs.

These findings provide a solid foundation for further statistical analysis, visualization, and modeling activities.